In [ ]:
# BSD 3-Clause License
#
# Copyright 2026 Renesas Electronics Corporation and/or its affiliates
# All rights reserved.
# 
# Redistribution and use in source and binary forms, with or without
# modification, are permitted provided that the following conditions are met:
# 
# 1. Redistributions of source code must retain the above copyright notice, this
#    list of conditions and the following disclaimer.
# 
# 2. Redistributions in binary form must reproduce the above copyright notice,
#    this list of conditions and the following disclaimer in the documentation
#    and/or other materials provided with the distribution.
# 
# 3. Neither the name of the copyright holder nor the names of its
#    contributors may be used to endorse or promote products derived from
#    this software without specific prior written permission.
# 
# THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"
# AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE
# IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE
# DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE
# FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL
# DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR
# SERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER
# CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY,
# OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE
# OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.
# 

# SPDX-License-Identifier: BSD-3-Clause

# CIFAR-10 Quantization Workflow with RUHMI'S AI Compiler for MCU
## Introduction
This tutorial demonstrates an end-to-end workflow for deploying a Deep Learning model to an Edge AI target using the **RUHMI Framework** AI MCU Compiler (accessed via the `mera` library).

### Objective
The primary goal is to take a standard floating-point model (FP32) and optimize it for a constrained microcontroller (MCU) environment using **INT8 Quantization**. We will perform a comparative analysis of four model variants to validate the quantization strategy:
1.  **TFLite FP32**: Baseline uncompressed model.
2.  **TFLite INT8**: Standard TensorFlow Lite quantization reference.
3.  **RUHMI AI Compiler FP32**: Bit-exact compilation of the original model (verifies compiler correctness).
4.  **Ruhmi AI Compiler INT8**: The target quantized model deployed via RUHMI's hardware-aware quantization flow.

> **Note:** RUHMI AI Compiler for MCU is powered by Mera from EdgeCortix

### Workflow Stages
1.  **Training**: Train a custom CNN on CIFAR-10.
2.  **Conversion**: Generate baseline TFLite artifacts.
3.  **Hardware-Aware Quantization**: Use Mera to quantize and optimize for MCU.
4.  **C-Code Deployment**: Compile the model into bit-exact C-code for host simulation.
5.  **Validation**: Comparative accuracy assessment.



## 0. Prerequisites & Setup

> **⚠️ Terminal Setup Required**
>
> All setup steps below must be performed in your **terminal** before launching this notebook.

### 1. Install MERA Environment
Follow the [Installation Guide](../../../install/README.md) to create your MERA virtual environment and install the compiler.

### 2. Install Tutorial Dependencies
With your `mera-env` activated, install the additional packages required by this tutorial:
```bash
pip install ipykernel scipy scikit-learn tensorflow matplotlib seaborn
```

### 3. Register Jupyter Kernel & Launch
```bash
python -m ipykernel install --user --name=mera-env --display-name "Python (MERA Env)"
jupyter notebook
```
**Note**: In the Jupyter interface, select **Kernel > Change Kernel > Python (MERA Env)**.

In [ ]:
# ----------------------------------------------------------------
# ENVIRONMENT VERIFICATION
# ----------------------------------------------------------------
import sys
import shutil
import importlib

print(f"Python Executable: {sys.executable}")

# 1. Check if we are running from a venv
is_venv = (sys.prefix != sys.base_prefix)
print(f"Running in Venv: {'✅ Yes' if is_venv else '❌ NO - WARN: Using system python!'}")

if not is_venv:
    print("\n⚠️ WARNING: You are NOT running inside a virtual environment.")
    print("   This often causes version conflicts (e.g. NumPy 1.x vs 2.x).")
    print("   Please ensure you selected the correct Kernel (e.g. 'Python (mera-env)').")

# 2. Verify MERA and NumPy
try:
    import mera
    import numpy

    print(f"\n✅ MERA version: {mera.__version__}")
    print(f"   NumPy version: {numpy.__version__}")
    
    if numpy.__version__.startswith('2.'):
        print("   ❌ ERROR: NumPy 2.x detected! MERA requires NumPy 1.x.")
    elif 'site-packages/mera' in mera.__file__ and '.local' in mera.__file__:
        print(f"   ❌ ERROR: Loading MERA from user site-packages: {mera.__file__}")
    else:
        print("   ✅ Environment looks good.")
        
except ImportError as e:
    print(f"\n❌ Import Error: {e}")
    print("   Check your kernel selection.")

# Verify CMake
if shutil.which('cmake'):
    print(f"\n✅ CMake found: {shutil.which('cmake')}")
else:
    print("\n❌ CMake not found. Please install via system package manager.")

In [ ]:
# Check GCC version (Required for RUHMI/MERA C-code generation)
!gcc --version | head -n 1
!g++ --version | head -n 1


> **ℹ️ Terminology: RUHMI vs MERA**
> *   **RUHMI Framework**: The comprehensive suite of tools for AI model deployment on the Edge. It encapsulates vairety of tools to ease deployment on edge devices.
> *   **MERA**: The Python library (developed by EdgeCortix) used *within* the RUHMI ecosystem specifically for the AI  Compiler and quantization tasks.
> This tutorial uses the MERA library to perform the compilation steps required by the RUHMI Framework.


## 1. Development Environment Configuration
We initialize the environment by setting up TensorFlow for model training and importing the `mera` library, which serves as the Python interface for the compiler. Note that warning suppression is enabled to maintain a clean log output for this demonstration.



In [ ]:
import os
import sys
import logging
import warnings

# Suppress warnings and logs
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')
logging.getLogger('tensorflow').setLevel(logging.FATAL)

import shutil
import subprocess
import pickle
import numpy as np
from pathlib import Path
from datetime import datetime

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib as plt

import mera
from mera import Target, Platform
from mera.mera_quantizer import quantizer

import matplotlib.pyplot as plt
import seaborn as sns


print(f"TensorFlow version: {tf.__version__}")
print(f"MERA version: {mera.__version__}")
print(f"NumPy version: {np.__version__}")

## 2. Global Configuration
Here we define the training hyperparameters and class outputs.
*   **Calibration Set**: 500 images are reserved for collecting activation statistics during Post-Training Quantization (PTQ).
*   **Test Set**: 1000 images are used for final accuracy verification.
*   **Epochs**:  Number of iterations the training is done.
*   **Test Set**: Number of input models fed into the model at once, training tends to be done faster with larger batch size at expense of RAM consumption.



In [ ]:
# Paths
OUTPUT_DIR = Path("CIFAR10_RUHMI_Tutorial")
OUTPUT_DIR.mkdir(exist_ok=True)

# Training config
EPOCHS = 15
BATCH_SIZE = 64

# Test config
NUM_CALIB_IMAGES = 500
NUM_TEST_IMAGES = 1000

# CIFAR-10 classes
CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']

print(f"Output directory: {OUTPUT_DIR.absolute()}")

## 3. Dataset Preparation (CIFAR-10)
We load the CIFAR-10 dataset and apply `[0, 1]` normalization.

> **🧠 Why [0, 1] Normalization?**
> While research models often use complex normalization (e.g., mean subtraction `(x - mean) / std`), we use simple `[0, 1]` scaling for deployment efficiency.
> *   **Hardware Alignment**: MCU accelerators often process unsigned 8-bit integers (0-255). Scaling floats to `[0, 1]` maps cleanly to this range without expensive pre-processing arithmetic on the device.
> *   **Consistency**: The preprocessing logic used here `MUST` match the C-code on the device exactly. Mismatched scaling is the #1 cause of "the model works in Python but fails on the board."


## Dataset
This example uses the CIFAR-10 dataset:
> Alex Krizhevsky, "Learning Multiple Layers of Features from Tiny Images", 2009
> https://www.cs.toronto.edu/~kriz/cifar.html

In [ ]:
# Load CIFAR-10
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

print(f"Training data: {x_train.shape}")
print(f"Test data: {x_test.shape}")
print(f"Pixel range: [{x_train.min()}, {x_train.max()}]")

# Normalize to [0, 1] range (simpler preprocessing)
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Flatten labels
y_train = y_train.flatten()
y_test = y_test.flatten()

print(f"\nAfter normalization:")
print(f"Pixel range: [{x_train.min():.3f}, {x_train.max():.3f}]")

### Data Visualization

The images are 32 by 32 pixels RGB, a few images are vizualized below.

In [ ]:
# Visualization
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

plt.figure(figsize=(10,10))
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(x_train[i])
    # The CIFAR labels happen to be arrays,
    # which is why you need the extra index
    plt.xlabel(class_names[y_train[i]])
plt.show()

## 4. Target Model Architecture
We define a VGG-style Convolutional Neural Network (CNN) tailored for edge efficiency.
*   **Structure**: 3 Convolutional blocks with Batch Normalization and ReLU activation.
*   **Complexity**: ~400k parameters, suitable for mid-range MCUs.



In [ ]:
def create_simple_cnn():
    """Create a simple CNN for CIFAR-10 classification."""
    model = keras.Sequential([
        # Input
        layers.Input(shape=(32, 32, 3)),
        
        # Conv Block 1
        layers.Conv2D(32, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Conv2D(32, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Conv Block 2
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Conv Block 3
        layers.Conv2D(128, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Classifier
        layers.Flatten(),
        layers.Dense(128),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ])
    
    return model

model = create_simple_cnn()
model.summary()

## 5. Training Reference Model
We train the FP32 reference model. A validation accuracy of >75% is expected, which provides a sufficient baseline to measure quantization impact.
Do note we are not using a GPU for this activity, but the model is simple enough to be trained on your own laptops.


### ⏰ Checkpoint: Pre-trained Model

To expedite this tutorial, you can download a pre-trained model (`.keras`) instead of training from scratch.
Set `USE_PRETRAINED_MODEL = True` in the cell below to verify your logic without waiting for training.

The model is trained on the training dataset and then evaluted on the testing dataset to ensure generalization.

In [ ]:
# Configuration: Set to True to skip training
USE_PRETRAINED_MODEL = False

if USE_PRETRAINED_MODEL:
    import urllib.request
    
    # Placeholder URL - Replace with specific model release URL
    MODEL_URL = "https://github.com/renesas/ruhmi-framework-mcu/releases/download/v2.5.0/cifar10_pretrained.keras"
    PRETRAINED_PATH = OUTPUT_DIR / "cifar10_pretrained.keras"
    
    print(f"Checking for pre-trained model...")
    if not PRETRAINED_PATH.exists():
        print(f"Downloading from {MODEL_URL}...")
        try:
            urllib.request.urlretrieve(MODEL_URL, PRETRAINED_PATH)
            print("✅ Download complete.")
        except Exception as e:
            print(f"❌ Download failed: {e}")
            print("Reverting to training mode.")
            USE_PRETRAINED_MODEL = False
            
    if USE_PRETRAINED_MODEL:
        print("Loading pre-trained model...")
        try:
            model = keras.models.load_model(PRETRAINED_PATH)
            print("✅ Model loaded successfully. Training will be skipped.")
        except Exception as e:
            print(f"❌ Failed to load model: {e}")
            USE_PRETRAINED_MODEL = False


In [ ]:
if not USE_PRETRAINED_MODEL:
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    # Data augmentation
    datagen = keras.preprocessing.image.ImageDataGenerator(
        rotation_range=15,
        width_shift_range=0.1,
        height_shift_range=0.1,
        horizontal_flip=True
    )
    datagen.fit(x_train)
    
    # Train
    print("Training...")
    history = model.fit(
        datagen.flow(x_train, y_train, batch_size=BATCH_SIZE),
        epochs=EPOCHS,
        validation_data=(x_test, y_test),
        verbose=1
    )
else:
    print("Skipping training (using pre-trained model).")


## Training Visualization: Accuracy & Loss

To evaluate the performance of our AI models during training, we primarily track two metrics: **Accuracy** and **Loss**.

###  The Metrics
* **Accuracy:** Measures how often the model's prediction matches the expected output.
    * *Goal:* **Higher is better.** (e.g., 90% accuracy means 9 out of 10 predictions are correct).
* **Loss:** Quantifies the "distance" or error between the model's prediction and the actual target. This acts as the error signal used to update the model's weights during backpropagation.
    * *Goal:* **Lower is better.**

### The Training Cycle (Epochs)
As the model iterates through the dataset (epochs), we look for a balance between learning patterns and memorizing data.



* **Underfitting (Too Little Training):**
    * *Symptom:* High Loss, Low Accuracy.
    * *Cause:* The model has not seen the data enough times to learn the underlying patterns. It fails to generalize because it hasn't learned the rules yet.

* **Overfitting (Too Much Training):**
    * *Symptom:* Training Accuracy is very high, but Validation Accuracy stalls or drops.
    * *Cause:* The model has reached its capacity and begun "memorizing" the specific noise or traits of the training data rather than learning general features. It works perfectly on known data but fails on new, unseen data.

* **The "Sweet Spot" (Generalization):**
    * *Goal:* We aim for the point where the Training Loss and Validation Loss are both low and stable. This indicates the model has learned the general rules without memorizing the specific answers.

In [ ]:
if 'history' in locals():
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Accuracy')
    plt.plot(history.history['val_accuracy'], label = 'Val Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.ylim([0.5, 1])
    plt.legend(loc='lower right')
    plt.title('Training & Validation Accuracy')

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Loss')
    plt.plot(history.history['val_loss'], label = 'Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend(loc='upper right')
    plt.title('Training & Validation Loss')
    plt.show()
else:
    print("Training skipped (pre-trained model used), history not available.")

### Baseline Evaluation
We save the Keras model (`.keras`) for subsequent ingestion by the quantization tools.

The testing dataset is now used to ensure generalization. Avoid using training or validation dataset for evaluation purposes.

In [ ]:
# Evaluate
print("\nEvaluating on test set...")
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc*100:.2f}%")

# Save model
KERAS_MODEL_PATH = OUTPUT_DIR / "cifar10_cnn.keras"
model.save(KERAS_MODEL_PATH)
print(f"\nSaved: {KERAS_MODEL_PATH}")

### Visualizing Errors: The Confusion Matrix

Accuracy tells you how often the model is right. The Confusion Matrix tells you how it is wrong. It compares Predictions vs. Reality.
The 4 States:

    True Positive (TP): Correct detection. (e.g., "Fault detected, and there was a fault.")
    True Negative (TN): Correct silence. (e.g., "No fault detected, system is normal.")
    False Positive (FP): False Alarm. (e.g., "Fault detected," but system is actually fine.)
    False Negative (FN): Missed Target. (e.g., "No fault detected," but the system is actually broken.)

Ideally, we want a diagonal matrix.

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Predict on test set
print("Generating predictions for Confusion Matrix...")
y_pred = model.predict(x_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = y_test.flatten()

# Compute confusion matrix
cm = confusion_matrix(y_true, y_pred_classes)

# Plot
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

## 6. Baseline Generation (TFLite)
To benchmark RUHMI's AI MCU compiler performance, we first generate standard TFLite models.
*   **FP32**: The unquantized flatbuffer.
*   **INT8**: Standard TFLite post-training quantization. This serves as the content-correctness reference.

Calibration involves running a small, representative dataset through the model to determine the statistical distribution (min/max range) of the activation values. This ensures we map our limited INT8 grid to exactly where the data lives, preserving the original FP32 signal shape as closely as possible.

>> **Note** Its important to use the same preprocessing for calibration data as used for model training to ensure the validity of the calibration.


In [ ]:
# Prepare calibration data
calib_data = x_train[:NUM_CALIB_IMAGES]
print(f"Calibration data: {calib_data.shape}")

# TFLite FP32
print("\nConverting to TFLite FP32...")
converter_fp32 = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_fp32 = converter_fp32.convert()

TFLITE_FP32_PATH = OUTPUT_DIR / "cifar10_fp32.tflite"
with open(TFLITE_FP32_PATH, 'wb') as f:
    f.write(tflite_fp32)
print(f"Saved: {TFLITE_FP32_PATH} ({len(tflite_fp32)/1024:.1f} KB)")



In [ ]:
# TFLite INT8
print("\nConverting to TFLite INT8...")
def representative_dataset():
    for img in calib_data:
        yield [np.expand_dims(img, axis=0).astype(np.float32)]

converter_int8 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_dataset

tflite_int8 = converter_int8.convert()

TFLITE_INT8_PATH = OUTPUT_DIR / "cifar10_int8.tflite"
with open(TFLITE_INT8_PATH, 'wb') as f:
    f.write(tflite_int8)
print(f"Saved: {TFLITE_INT8_PATH} ({len(tflite_int8)/1024:.1f} KB)")

print(f"\nCompression: {len(tflite_fp32)/len(tflite_int8):.2f}x smaller")

### Vizualization of quantization 

AI models tend to be quite redundant, quantization enables reduction in precision with minimal sacrifice to accuracy.  

FP32 is sort of like a continous analog sensor signal  
INT8 is the 8-bit ADC reading


In [ ]:
# -----------------------------------------------------------
# VISUALIZATION: Weight Distribution (FP32 vs INT8)
# -----------------------------------------------------------
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def get_weights_from_tflite(tflite_path):
    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()
    details = interpreter.get_tensor_details()
    
    # Extract weights from the first Conv2D layer we find
    # This is heuristic but works for standard CNNs like ours
    for tensor in details:
        if 'Conv2D' in tensor['name'] and 'W' in tensor['name']:
             # 'W' usually denotes weights in some TFLite schemas, checking shape is safer
             # Heuristic: 4D tensor is usually a Conv kernel
             if len(tensor['shape']) == 4:
                 return interpreter.get_tensor(tensor['index']).flatten()
    # Fallback: Just grab the largest tensor
    largest_idx = max(range(len(details)), key=lambda i: np.prod(details[i]['shape']))
    return interpreter.get_tensor(details[largest_idx]['index']).flatten()

try:
    fp32_weights = get_weights_from_tflite(TFLITE_FP32_PATH)
    int8_weights = get_weights_from_tflite(TFLITE_INT8_PATH)

    plt.figure(figsize=(12, 5))

    # Plot FP32
    plt.subplot(1, 2, 1)
    sns.histplot(fp32_weights, kde=True, color='blue', bins=50)
    plt.title("FP32 Weight Distribution (Continuous)")
    plt.xlabel("Weight Value")
    plt.ylabel("Count")

    # Plot INT8
    plt.subplot(1, 2, 2)
    sns.histplot(int8_weights, kde=False, color='red', bins=50)
    plt.title("INT8 Weight Distribution (Discrete)")
    plt.xlabel("Quantized Integer Value")
    plt.ylabel("Count")

    plt.tight_layout()
    plt.show()

    print("✅ Visualization Complete: Notice how the smooth FP32 curve becomes discrete bars in INT8.")
except Exception as e:
    print(f"⚠️ Could not generate visualization: {e}")
    print("Ensure TFLite files exist and paths are correct.")


## 7. RUHMI Compiler Verification (FP32)
Before quantizing, we verified the compilation path using mera.
We use `mera.Deployer` to generate C-code from the FP32 model.
*   **`use_x86=True`**: This flag instructs the compiler to generate a CMake project compatible with the host (x86) architecture, enabling us to run the generated layer-code locally as a Python extension (`py_compute`). This allows for bit-exact simulation of the MCU runtime on a PC.



In [ ]:
def build_mera_ccode(mcu_dir):
    """Build MERA C-code and return py_compute directory."""
    cmake_files = list(mcu_dir.rglob("CMakeLists.txt"))
    if not cmake_files:
        raise FileNotFoundError(f"CMakeLists.txt not found in {mcu_dir}")
    
    source_dir = cmake_files[0].parent
    build_dir = source_dir / "build"
    
    if build_dir.exists():
        shutil.rmtree(build_dir)
    build_dir.mkdir(parents=True)
    
    # Configure
    result = subprocess.run(
        ["cmake", "-DBUILD_PY_BINDINGS=ON", ".."],
        cwd=build_dir, capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f"CMake failed: {result.stderr}")
    
    # Build
    result = subprocess.run(
        ["cmake", "--build", ".", "--parallel", "4"],
        cwd=build_dir, capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f"Build failed: {result.stderr}")
    
    # Find py_compute
    py_compute_paths = list(build_dir.rglob("py_compute*.so")) + \
                       list(build_dir.rglob("py_compute*.pyd"))
    if not py_compute_paths:
        raise FileNotFoundError(f"py_compute not found")
    
    return str(py_compute_paths[0].parent)

In [ ]:
# Deploy MERA FP32 to C-code
print("Deploying MERA FP32 to C-code...")

MERA_FP32_DIR = OUTPUT_DIR / "mera_fp32"
with mera.Deployer(str(MERA_FP32_DIR), overwrite=True) as deployer:
    mera_model = mera.ModelLoader(deployer).from_tflite(str(TFLITE_FP32_PATH))
    mcu_config = {'use_x86': True, 'suffix': '', 'weight_location': 'flash'}
    deployer.deploy(mera_model, mera_platform=Platform.MCU_CPU, target=Target.MCU, mcu_config=mcu_config)

print("Building MERA FP32 C-code...")
MERA_FP32_PYCOMPUTE = build_mera_ccode(MERA_FP32_DIR)
print(f"✅ MERA FP32 C-code built: {MERA_FP32_PYCOMPUTE}")

## 8. Mera Quantization (INT8)
This is the core optimization step. We instantiate the quantization via `mera.Quantizer`.
*   **Calibration**: We pass a representative dataset to the quantizer to observe activation ranges.
*   **QuantizerConfigPresets.MCU**: This preset applies hardware-aware constraints ideal for Cortex-M targets 



In [ ]:
# Quantize with MERA
print("Quantizing with MERA...")

MERA_QTZ_DIR = OUTPUT_DIR / "mera_quantization"
with mera.Deployer(str(MERA_QTZ_DIR), overwrite=True) as deployer:
    mera_model = mera.ModelLoader(deployer).from_tflite(str(TFLITE_FP32_PATH))
    input_name = list(mera_model.input_desc.all_inputs.keys())[0]
    
    # Prepare calibration data
    mera_calib_data = [
        {input_name: np.expand_dims(img, axis=0).astype(np.float32)}
        for img in calib_data[:100]
    ]
    
    print(f"Calibrating with {len(mera_calib_data)} samples...")
    qtzer = mera.Quantizer(
        deployer, mera_model,
        quantizer_config=quantizer.QuantizerConfigPresets.MCU,
        mera_platform=Platform.MCU_CPU
    )
    
    result = qtzer.calibrate(mera_calib_data).quantize().evaluate_quality(mera_calib_data[:1])
    Q = result[0].out_summary()[0]
    print(f"Quantization quality: PSNR={Q['psnr']:.2f}, Score={Q['score']:.4f}")
    
    MERA_QTZ_PATH = MERA_QTZ_DIR / "model.mera"
    qtzer.save_to(str(MERA_QTZ_PATH))
    print(f"Saved: {MERA_QTZ_PATH}")

In [ ]:
input_name = list(mera_model.input_desc.all_inputs.keys())[0]

### Deploying Quantized Model
Similar to the FP32 step, we compile the now-quantized `model.mera` artifact into executable C-code.



In [ ]:
# Deploy MERA INT8 to C-code
print("Deploying MERA INT8 to C-code...")

MERA_INT8_DIR = OUTPUT_DIR / "mera_int8"
with mera.Deployer(str(MERA_INT8_DIR), overwrite=True) as deployer:
    int8_model = mera.ModelLoader(deployer).from_quantized_mera(str(MERA_QTZ_PATH))
    mcu_config = {'use_x86': True, 'suffix': '', 'weight_location': 'flash'}
    deployer.deploy(int8_model, mera_platform=Platform.MCU_CPU, target=Target.MCU, mcu_config=mcu_config)

print("Building MERA INT8 C-code...")
MERA_INT8_PYCOMPUTE = build_mera_ccode(MERA_INT8_DIR)
print(f"✅ MERA INT8 C-code built: {MERA_INT8_PYCOMPUTE}")

## 9. Comprehensive Accuracy Validation
We now have four distinct model artifacts. We will execute inference on the same test set (1000 images) across all four to verify performance and correctness.
*   **Methodology**: For the mera path, we import the generated `py_compute` module and execute the C-function `compute()` directly.



In [ ]:
def run_mera_inference(py_compute_dir, test_images):
    """Run MERA C-code inference via subprocess."""
    # Save test data
    data_file = "/tmp/mera_test_data.pkl"
    result_file = "/tmp/mera_results.pkl"
    
    with open(data_file, 'wb') as f:
        pickle.dump(test_images, f)
    
    script = f'''
import sys
import pickle
import numpy as np
sys.path.insert(0, "{py_compute_dir}")
import py_compute as c

with open("{data_file}", "rb") as f:
    data = pickle.load(f)

preds = []
for img in data:
    inp = np.expand_dims(img, axis=0).astype(np.float32)
    out = c.compute(inp)[0].flatten()
    preds.append(int(np.argmax(out)))

with open("{result_file}", "wb") as f:
    pickle.dump(preds, f)
'''
    result = subprocess.run(
        [sys.executable, "-c", script],
        capture_output=True, text=True
    )
    
    if result.returncode != 0:
        print(f"Error: {result.stderr}")
        return None
    
    with open(result_file, 'rb') as f:
        return pickle.load(f)

In [ ]:
# Prepare test data
test_images = x_test[:NUM_TEST_IMAGES]
test_labels = y_test[:NUM_TEST_IMAGES]

print(f"Testing with {len(test_images)} images...")

results = {}

# 1. TFLite FP32
print("\n1. Testing TFLite FP32...")
tflite_fp32_interp = tf.lite.Interpreter(model_path=str(TFLITE_FP32_PATH))
tflite_fp32_interp.allocate_tensors()
inp_det = tflite_fp32_interp.get_input_details()[0]
out_det = tflite_fp32_interp.get_output_details()[0]

tflite_fp32_preds = []
for img in test_images:
    tflite_fp32_interp.set_tensor(inp_det['index'], np.expand_dims(img, axis=0).astype(np.float32))
    tflite_fp32_interp.invoke()
    out = tflite_fp32_interp.get_tensor(out_det['index']).flatten()
    tflite_fp32_preds.append(int(np.argmax(out)))

results['TFLite FP32'] = {
    'preds': tflite_fp32_preds,
    'acc': sum(p == l for p, l in zip(tflite_fp32_preds, test_labels)) / len(test_labels) * 100
}
print(f"   Accuracy: {results['TFLite FP32']['acc']:.2f}%")

In [ ]:
# 2. TFLite INT8
print("2. Testing TFLite INT8...")
tflite_int8_interp = tf.lite.Interpreter(model_path=str(TFLITE_INT8_PATH))
tflite_int8_interp.allocate_tensors()
inp_det = tflite_int8_interp.get_input_details()[0]
out_det = tflite_int8_interp.get_output_details()[0]

tflite_int8_preds = []
for img in test_images:
    tflite_int8_interp.set_tensor(inp_det['index'], np.expand_dims(img, axis=0).astype(np.float32))
    tflite_int8_interp.invoke()
    out = tflite_int8_interp.get_tensor(out_det['index']).flatten()
    tflite_int8_preds.append(int(np.argmax(out)))

results['TFLite INT8'] = {
    'preds': tflite_int8_preds,
    'acc': sum(p == l for p, l in zip(tflite_int8_preds, test_labels)) / len(test_labels) * 100
}
print(f"   Accuracy: {results['TFLite INT8']['acc']:.2f}%")

In [ ]:
# 3. MERA FP32 C-code
print("3. Testing MERA FP32 C-code...")
mera_fp32_preds = run_mera_inference(MERA_FP32_PYCOMPUTE, test_images)

if mera_fp32_preds:
    results['MERA FP32'] = {
        'preds': mera_fp32_preds,
        'acc': sum(p == l for p, l in zip(mera_fp32_preds, test_labels)) / len(test_labels) * 100
    }
    print(f"   Accuracy: {results['MERA FP32']['acc']:.2f}%")
else:
    print("   ❌ Failed to run MERA FP32")

In [ ]:
# 4. MERA INT8 C-code
print("4. Testing MERA INT8 C-code...")
mera_int8_preds = run_mera_inference(MERA_INT8_PYCOMPUTE, test_images)

if mera_int8_preds:
    results['MERA INT8'] = {
        'preds': mera_int8_preds,
        'acc': sum(p == l for p, l in zip(mera_int8_preds, test_labels)) / len(test_labels) * 100
    }
    print(f"   Accuracy: {results['MERA INT8']['acc']:.2f}%")
else:
    print("   ❌ Failed to run MERA INT8")

## 10. Comparative Analysis Results
The table below summarizes the accuracy drop relative to the TFLite FP32 baseline.

### 📊 How to Interpret These Numbers
*   **Accuracy Retention**: A good quantized model should stay within **1-2%** of the FP32 baseline.
*   **Sanity Check (Unique Preds)**: We check distinct prediction counts. A "collapsed" model might output "Class 0" for every single image and still get 10% accuracy (random guessing). Seeing 10/10 unique predictions confirms the model is actually evaluating features.
*   **Bit-Exact Verification**: `MERA FP32` should match `TFLite FP32`. This confirms the compiler is mathematically sound before we even look at quantization. Though in this tutorial we are using Tflite interpreter whereas for Mera its the actual code run via Cmake.


In [ ]:
print("=" * 70)
print("ACCURACY COMPARISON RESULTS")
print("=" * 70)
print(f"\nTest set: {NUM_TEST_IMAGES} images from CIFAR-10")
print(f"Calibration: {NUM_CALIB_IMAGES} images")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

baseline = results['TFLite FP32']['acc']

print("\n" + "-" * 70)
print(f"{'Model':<25} {'Accuracy':>12} {'Unique Preds':>15} {'Drop':>12}")
print("-" * 70)

for name in ['TFLite FP32', 'TFLite INT8', 'MERA FP32', 'MERA INT8']:
    if name not in results:
        print(f"{name:<25} {'N/A':>12} {'N/A':>15} {'N/A':>12}")
        continue
        
    r = results[name]
    acc = r['acc']
    unique = len(set(r['preds']))
    drop = acc - baseline
    drop_str = f"{drop:+.2f}%" if name != 'TFLite FP32' else "---"
    status = "✅" if unique >= 5 else "❌ BROKEN"
    
    print(f"{name:<25} {acc:>11.2f}% {unique:>14} {drop_str:>12} {status}")

print("-" * 70)

## 11. Conclusion
*   **Optimization Success**: If the MERA INT8 drop is acceptable, the model is ready for MCU deployment.
*   **Verification**: The bit-exact host simulation confirms that the C-code logic is sound.



In [ ]:
import os

# -----------------------------------------------------------
# FINAL REPORT: Accuracy & Size Benchmark
# -----------------------------------------------------------
print("\n" + "=" * 60)
print(f"{'QUANTIZATION BENCHMARK SUMMARY':^60}")
print("=" * 60)

# --- 1. ACCURACY CHECK ---
tflite_drop = results['TFLite INT8']['acc'] - baseline

if 'MERA INT8' in results:
    mera_drop = results['MERA INT8']['acc'] - baseline
    mera_unique = len(set(results['MERA INT8']['preds']))
    
    print(f"\n{ 'ACCURACY':<20} {'Drop from Baseline':>20}")
    print("-" * 42)
    print(f"{'TFLite INT8':<20} {tflite_drop:>19.2f}%")
    print(f"{'MERA INT8':<20} {mera_drop:>19.2f}%")
    
    print("\nStatus:")
    if mera_unique >= 5 and abs(mera_drop) < 10:
        print("  ✅ MERA quantization is working correctly!")
    elif mera_unique < 5:
        print(f"  ❌ MERA INT8 is BROKEN - only {mera_unique} unique predictions")
        print(f"     Always predicts: {sorted(set(results['MERA INT8']['preds']))}")
    else:
        print(f"  ⚠️ MERA INT8 has significant accuracy drop: {mera_drop:.1f}%")
else:
    print("\n❌ MERA INT8 test failed to run.")

# --- 2. SIZE CHECK ---
fp32_size = os.path.getsize(TFLITE_FP32_PATH) / 1024
int8_size = os.path.getsize(TFLITE_INT8_PATH) / 1024
reduction = (fp32_size - int8_size) / fp32_size * 100
savings = fp32_size - int8_size

print("\n" + "-" * 60)
print(f"{ 'MODEL SIZE':<20} {'Size (KB)':>20}")
print("-" * 60)
print(f"{'TFLite FP32':<20} {fp32_size:>19.1f} KB")
print(f"{'TFLite INT8':<20} {int8_size:>19.1f} KB")
print("-" * 60)
print(f"Space Savings: {reduction:.1f}% reduction")
print(f"Flash Saved:   {savings:.1f} KB")
print("=" * 60)